In [18]:
import numpy as np
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern
from scipy.stats import norm
from scipy.optimize import minimize

In [19]:
# Define the target function
def target_function(x):
    return np.sum(x)

In [20]:
# Acquisition function: Expected Improvement
def acquisition_function(X, X_sample, y_sample, gpr, xi=0.01):
    mu, sigma = gpr.predict(X, return_std=True)
    mu_sample = gpr.predict(X_sample)

    sigma = sigma.reshape(-1, 1)
    mu_sample_opt = np.max(mu_sample)

    with np.errstate(divide='warn'):
        imp = mu - mu_sample_opt - xi
        Z = imp / sigma
        ei = imp * norm.cdf(Z) + sigma * norm.pdf(Z)
        ei[sigma == 0.0] = 0.0

    return ei

In [21]:
# Bayesian optimization
def bayesian_optimization(n_iter, bounds, n_init=1000):
    # Initialize the Gaussian process regressor
    kernel = Matern(nu=2.5)
    gpr = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=10)

    # Generate initial samples
    X_sample = np.random.uniform(bounds[:, 0], bounds[:, 1], size=(n_init, bounds.shape[0]))
    y_sample = np.array([target_function(x) for x in X_sample])

    for _ in range(n_iter):
        # Fit the Gaussian process to the current data
        gpr.fit(X_sample, y_sample)

        # Define the objective function for the acquisition function optimization
        def min_obj(x):
            return -acquisition_function(x.reshape(1, -1), X_sample, y_sample, gpr)

        # Find the next point to sample
        res = minimize(min_obj, x0=np.random.uniform(bounds[:, 0], bounds[:, 1], size=bounds.shape[0]),
                       bounds=bounds)
        next_point = res.x

        # Evaluate the target function at the next point
        next_value = target_function(next_point)

        # Update the samples
        X_sample = np.vstack((X_sample, next_point))
        y_sample = np.hstack((y_sample, next_value))

    # Find the maximum value and the corresponding input
    max_index = np.argmax(y_sample)
    max_value = y_sample[max_index]
    max_input = X_sample[max_index]

    return max_value, max_input

In [22]:
# Define the bounds for each driver
bounds = np.array([[1, 100]] * 6)

# Run the Bayesian optimization
n_iter = 10
max_value, max_input = bayesian_optimization(n_iter, bounds)

print(f"Maximum value: {max_value}")
print(f"Optimal input: {max_input}")

Maximum value: 496.5401618191096
Optimal input: [95.0697984  81.03514464 62.67382539 80.8146093  97.22048743 79.72629666]
